# ATLAS simulator: theory notebooks

These notebooks walk through the physics inside `airframe_designer/`, the simulator that flies the ATLAS V3 model against PX4 (SITL and HITL). Each notebook covers one aspect. Every important code cell has the theory above it and a short **Note** in plain words, and the code calls the simulator's real functions on the real V3 airframe, so the numbers you see are the ones the simulator uses.

| Notebook | Aspect | Main code it uses |
|---|---|---|
| [01_frames](01_frames.ipynb) | World/inertial frame, structural, hover, rotor and sensor frames, pitch rotations | `geometry/airframe.py`, `geometry/frames.py` |
| [02_quaternions](02_quaternions.ipynb) | Unit quaternions, Hamilton product, attitude kinematics, the 90° singularity | `dynamics/quaternion.py` |
| [03_equations_of_motion](03_equations_of_motion.ipynb) | State vector, Newton and Euler equations, inertia and the parallel-axis theorem, fan spool lag, the integrator | `dynamics/rigid_body.py`, `geometry/mass.py`, `aero/rotor_aero.py` |
| [04_ground_contact](04_ground_contact.ipynb) | Legs as algebraic penalty springs, friction, parking angle | `dynamics/contact.py`, `geometry/gear.py` |
| [05_sensors](05_sensors.ipynb) | Specific force, gyro, noise, the H-FLOW model and its EKF2 settings | `sensors/models.py`, `sensors/flow.py` |
| [06_fans_jetfoils_coanda](06_fans_jetfoils_coanda.ipynb) | Fan thrust, jetfoil turning, turning loss, ram drag, ducted-fan momentum theory, Coanda flow, the PX4 mixer export | `geometry/propulsion.py`, `aero/rotor_aero.py`, `geometry/airframe.py` |
| [07_jsbsim_and_chrono](07_jsbsim_and_chrono.ipynb) | JSBSim as a second physics backend through Python (`FGFDMExec`), and where Project Chrono would fit | `dynamics/jsbsim_backend.py` |

**Recommended order:** 01 → 02 → 03 → 04 → 05 → 06 → 07. Each one can also be run on its own.

## How to open and run them

- In **VS Code**: open a notebook and pick the kernel **AIRFRAME_DESIGNER (.venv)**.
- In **Jupyter Lab**: `uv pip install --python .venv/bin/python jupyterlab`, then `.venv/bin/jupyter lab notebooks/`.

Every notebook starts with the same setup cell. It adds the repository to the Python path and loads `airframes/atlas_v3_v34_foils_50_65_50_tuned.json` (the V3 model with 50 / 65 / 50° foils and the tuned gains on the board).

The companion report with the same material in long form is `docs/ATLAS_Simulator_Physics.pdf`.

In [1]:
# Setup: make the repository importable and load the V3 airframe used throughout these notebooks
import sys, math, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from airframe_designer.geometry.airframe import Airframe

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3})

AF_PATH = ROOT / "airframes" / "atlas_v3_v34_foils_50_65_50_tuned.json"
af = Airframe.load(AF_PATH)
af.resolve_mass()
G = 9.80665
print(f"{af.name}: {af.mass.mass:.2f} kg, {len(af.active_rotors())} fans, hover pitch {af.hover_pitch_deg} deg, parked {af.landed_pitch_deg} deg")

ATLAS_V3_V34_FOILS_50_65_50_TUNED: 11.83 kg, 9 fans, hover pitch 23.85 deg, parked 23.85 deg


## Conventions used everywhere

| Symbol | Meaning |
|---|---|
| $N$ superscript | components in the world NED frame (North, East, Down) |
| $B$ superscript | components in the body (structural, FRD) frame |
| $\mathbf q = [w, x, y, z]$ | attitude quaternion, scalar first, Hamilton product, body → NED |
| $R(\mathbf q)$ | rotation matrix, $\mathbf v^N = R\,\mathbf v^B$ |
| $R_h$ | structural → hover (PX4 body) frame, a pitch of $\theta_h$ |
| $\boldsymbol\omega^B = (p, q, r)$ | body angular velocity, rad/s |
| $\bar\Omega_i$ | fan $i$ speed as a fraction of full, 0 to 1 |
| $\Delta t$ | physics step, 1 ms |